<a href="https://colab.research.google.com/github/kietkorangicsmsrm515-sketch/12345/blob/main/truthlock_index_html.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install fastapi uvicorn python-multipart pillow requests

In [4]:
import os
import io
import json
import uuid
import hashlib
import sqlite3

from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

from PIL import Image, UnidentifiedImageError
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware

In [5]:
PROJECT_NAME = "TRUTHLOCK"

DATA_DIR = Path("/content/truthlock_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "truth_ledger.db"

MAX_FILE_SIZE = 25 * 1024 * 1024  # 25 MB

ALLOWED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".mp3", ".wav", ".mp4"
}

print("Project:", PROJECT_NAME)
print("Database:", DB_PATH)
print("Configuration ready!")

Project: TRUTHLOCK
Database: /content/truthlock_data/truth_ledger.db
Configuration ready!


In [6]:
def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database():
    with get_connection() as conn:

        conn.execute("""
            CREATE TABLE IF NOT EXISTS media_records (
                record_id TEXT PRIMARY KEY,
                filename TEXT NOT NULL,
                media_type TEXT NOT NULL,
                sha256 TEXT NOT NULL UNIQUE,
                file_size INTEGER NOT NULL,
                metadata_json TEXT NOT NULL,
                registered_at TEXT NOT NULL,
                source_description TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS evidence_events (
                event_id INTEGER PRIMARY KEY AUTOINCREMENT,
                record_id TEXT,
                event_type TEXT NOT NULL,
                event_details TEXT NOT NULL,
                event_timestamp TEXT NOT NULL
            )
        """)


initialize_database()

print("Database initialized successfully!")

Database initialized successfully!


In [7]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def calculate_sha256(content):
    return hashlib.sha256(content).hexdigest()


def detect_media_type(content, filename):
    ext = Path(filename).suffix.lower()

    if content.startswith(b"\xff\xd8\xff"):
        return ("image", ext in {".jpg", ".jpeg"})

    if content.startswith(b"\x89PNG\r\n\x1a\n"):
        return ("image", ext == ".png")

    if (
        len(content) >= 12
        and content[:4] == b"RIFF"
        and content[8:12] == b"WAVE"
    ):
        return ("audio", ext == ".wav")

    if (
        content.startswith(b"ID3")
        or (
            len(content) >= 2
            and content[0] == 0xFF
            and (content[1] & 0xE0) == 0xE0
        )
    ):
        return ("audio", ext == ".mp3")

    if len(content) >= 12 and content[4:8] == b"ftyp":
        return ("video", ext == ".mp4")

    return (None, False)


def extract_metadata(content, media_type):
    result = {
        "media_type": media_type,
        "file_size_bytes": len(content)
    }

    if media_type == "image":
        try:
            with Image.open(io.BytesIO(content)) as image:
                result.update({
                    "format": image.format,
                    "width": image.width,
                    "height": image.height,
                    "color_mode": image.mode,
                    "exif_present": bool(image.getexif())
                })

        except (UnidentifiedImageError, OSError):
            result["metadata_status"] = "Could not extract image metadata"

    else:
        result["metadata_status"] = (
            "Detailed audio/video metadata extraction is not implemented"
        )

    return result


def log_event(record_id, event_type, details):
    with get_connection() as conn:
        conn.execute("""
            INSERT INTO evidence_events
            (record_id, event_type, event_details, event_timestamp)
            VALUES (?, ?, ?, ?)
        """, (
            record_id,
            event_type,
            json.dumps(details),
            utc_now()
        ))


def get_record(record_id):
    with get_connection() as conn:
        row = conn.execute(
            "SELECT * FROM media_records WHERE record_id = ?",
            (record_id,)
        ).fetchone()

    return dict(row) if row else None


print("Helper functions ready!")

Helper functions ready!


In [8]:
app = FastAPI(
    title="TRUTHLOCK API",
    description="Deepfake Forensic Registry & Truth Ledger",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "http://localhost:5173"
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"]
)


@app.get("/")
def home():
    return {
        "project": PROJECT_NAME,
        "status": "running",
        "features": [
            "Media analysis",
            "SHA-256 fingerprinting",
            "Media registration",
            "File verification",
            "Evidence timeline"
        ],
        "ai_detection": "Not implemented",
        "blockchain": "Not implemented",
        "documentation": "/docs"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "timestamp": utc_now()
    }


print("FastAPI application created!")

FastAPI application created!


In [9]:
@app.post("/analyze")
async def analyze_media(file: UploadFile = File(...)):

    if not file.filename:
        raise HTTPException(400, "Filename required")

    filename = Path(file.filename).name

    content = await file.read(MAX_FILE_SIZE + 1)

    if not content:
        raise HTTPException(400, "Empty file")

    if len(content) > MAX_FILE_SIZE:
        raise HTTPException(413, "Maximum file size is 25 MB")

    media_type, extension_matches = detect_media_type(
        content, filename
    )

    if media_type is None or not extension_matches:
        raise HTTPException(
            415,
            "Unsupported file or extension/type mismatch"
        )

    fingerprint = calculate_sha256(content)

    metadata = extract_metadata(content, media_type)

    with get_connection() as conn:
        existing = conn.execute(
            "SELECT record_id FROM media_records WHERE sha256 = ?",
            (fingerprint,)
        ).fetchone()

    return {
        "filename": filename,
        "media_type": media_type,
        "sha256": fingerprint,
        "metadata": metadata,
        "registry_match": existing is not None,
        "matching_record_id": (
            existing["record_id"] if existing else None
        ),
        "ai_analysis": "Not implemented",
        "blockchain_verification": "Not implemented",
        "warning": "A hash does not prove that media is genuine."
    }


print("Media analyzer endpoint created!")

Media analyzer endpoint created!


In [10]:
@app.post("/register")
async def register_media(
    file: UploadFile = File(...),
    source_description: Optional[str] = None
):

    if not file.filename:
        raise HTTPException(400, "Filename required")

    filename = Path(file.filename).name

    content = await file.read(MAX_FILE_SIZE + 1)

    if not content:
        raise HTTPException(400, "Empty file")

    if len(content) > MAX_FILE_SIZE:
        raise HTTPException(413, "Maximum file size is 25 MB")

    media_type, extension_matches = detect_media_type(
        content, filename
    )

    if media_type is None or not extension_matches:
        raise HTTPException(
            415,
            "Unsupported file or extension/type mismatch"
        )

    fingerprint = calculate_sha256(content)
    metadata = extract_metadata(content, media_type)

    record_id = "TLR-" + uuid.uuid4().hex[:12].upper()
    registered_at = utc_now()

    try:
        with get_connection() as conn:
            conn.execute("""
                INSERT INTO media_records (
                    record_id, filename, media_type, sha256,
                    file_size, metadata_json, registered_at,
                    source_description
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                record_id,
                filename,
                media_type,
                fingerprint,
                len(content),
                json.dumps(metadata),
                registered_at,
                source_description
            ))

    except sqlite3.IntegrityError:
        raise HTTPException(
            409,
            "This exact file is already registered"
        )

    log_event(
        record_id,
        "MEDIA_REGISTERED",
        {
            "sha256": fingerprint,
            "source_identity_verified": False,
            "blockchain_anchored": False
        }
    )

    return {
        "message": "Media registered successfully",
        "record_id": record_id,
        "filename": filename,
        "sha256": fingerprint,
        "registered_at": registered_at,
        "metadata": metadata,
        "source_identity_verified": False,
        "blockchain_status": "Not implemented"
    }


print("Registration endpoint created!")

Registration endpoint created!


In [11]:
@app.post("/verify/{record_id}")
async def verify_media(
    record_id: str,
    file: UploadFile = File(...)
):

    record = get_record(record_id)

    if record is None:
        raise HTTPException(404, "Record not found")

    if not file.filename:
        raise HTTPException(400, "Filename required")

    filename = Path(file.filename).name
    content = await file.read(MAX_FILE_SIZE + 1)

    if not content:
        raise HTTPException(400, "Empty file")

    if len(content) > MAX_FILE_SIZE:
        raise HTTPException(413, "Maximum file size is 25 MB")

    media_type, extension_matches = detect_media_type(
        content, filename
    )

    if media_type is None or not extension_matches:
        raise HTTPException(
            415,
            "Unsupported file or extension/type mismatch"
        )

    uploaded_hash = calculate_sha256(content)
    matches = uploaded_hash == record["sha256"]

    log_event(
        record_id,
        "FILE_MATCHED" if matches else "FILE_DID_NOT_MATCH",
        {
            "uploaded_sha256": uploaded_hash,
            "hashes_match": matches
        }
    )

    return {
        "record_id": record_id,
        "hashes_match": matches,
        "result": (
            "VERIFIED_MATCH"
            if matches
            else "NO_MATCH_WITH_REGISTERED_VERSION"
        ),
        "explanation": (
            "The file matches the registered file byte for byte."
            if matches
            else "The file does not match the registered version."
        ),
        "important_note": (
            "A mismatch does not prove that the file is fake."
        )
    }


@app.get("/records")
def list_records():

    with get_connection() as conn:
        rows = conn.execute("""
            SELECT record_id, filename, media_type, sha256,
                   file_size, registered_at, source_description
            FROM media_records
            ORDER BY registered_at DESC
            LIMIT 100
        """).fetchall()

    return {
        "count": len(rows),
        "records": [dict(row) for row in rows]
    }


@app.get("/records/{record_id}")
def read_record(record_id: str):

    record = get_record(record_id)

    if record is None:
        raise HTTPException(404, "Record not found")

    record["metadata"] = json.loads(
        record.pop("metadata_json")
    )

    return {"record": record}


print("Verification and registry endpoints created!")

Verification and registry endpoints created!


In [12]:
@app.get("/timeline/{record_id}")
def get_timeline(record_id: str):

    if get_record(record_id) is None:
        raise HTTPException(404, "Record not found")

    with get_connection() as conn:
        rows = conn.execute("""
            SELECT event_id, record_id, event_type,
                   event_details, event_timestamp
            FROM evidence_events
            WHERE record_id = ?
            ORDER BY event_id ASC
        """, (record_id,)).fetchall()

    events = []

    for row in rows:
        event = dict(row)
        event["event_details"] = json.loads(
            event["event_details"]
        )
        events.append(event)

    return {
        "record_id": record_id,
        "events": events,
        "note": "Database event log; not an immutable blockchain ledger."
    }


print("All endpoints are ready!")

All endpoints are ready!


In [15]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)
server_thread.start()

print("TRUTHLOCK backend started on port 8000")

TRUTHLOCK backend started on port 8000


In [16]:
import requests
import time

time.sleep(2)

response = requests.get("http://127.0.0.1:8000/")

print("Status:", response.status_code)
print(response.json())

INFO:     127.0.0.1:34686 - "GET / HTTP/1.1" 200 OK
Status: 200
{'project': 'TRUTHLOCK', 'status': 'running', 'features': ['Media analysis', 'SHA-256 fingerprinting', 'Media registration', 'File verification', 'Evidence timeline'], 'ai_detection': 'Not implemented', 'blockchain': 'Not implemented', 'documentation': '/docs'}
